In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/pre_cell_26.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['sample_discourse_id']
me:  2
trying: ['train_first_last']
me:  11
trying: ['get_n_grams_2']
me:  25
trying: ['warnings']
me:  0
trying: ['other_words_to_take_out']
me:  22
trying: ['counts_dict']
me:  26
trying: ['cols_to_merge']
me:  13
trying: ['test_names']
me:  26
trying: ['f']
me:  26
trying: ['train']
me:  22
trying: ['counter']
me:  11
trying: ['cols_to_display']
me:  14
trying: ['last_ones']
me:  14
trying: ['stop_english']
me:  22
trying: ['text']
me:  22
trying: ['data']
me:  12
trying: ['bigrams']
me:  23
trying: ['test_txt']
me:  1
trying: ['ax2']
me:  8
trying: ['train_last']
me:  11
trying: ['word_dict']
me:  12
trying: ['get_n_grams']
me:  23
trying: ['len_dict']
me:  12
trying: ['train_texts']
me:  26
trying: ['style']
me:  0
trying: ['tqdm']
me:  0
trying: ['av_per_essay']
me:  8
trying: ['all_gaps']
me:  20
trying: ['glob']
me:  0
trying: ['add_gap_rows']
me:  20
trying: ['stopword

In [4]:
%%RecordEvent
%%time
### cell 26 ###

# Optimized cell 26
# 0. Limit to first 5 rows using head (avoids repeated __getitem__ calls)
train_text_df = train_text_df.head(5)

# 1. Tokenize each essay into one row per token, with token indices (all GPU ops)
tokenized = (
    train_text_df[['id', 'text']]
    .assign(tokens=lambda df: df['text'].str.split(' '))
    .explode('tokens')
    .reset_index(drop=True)
)
tokenized = tokenized.assign(
    token_index=lambda df: df.groupby('id').cumcount()
)

# 2. Expand discourse annotations into one row per token index with B-/I- labels
labels = (
    train[['id', 'discourse_type', 'predictionstring']]
    .reset_index()
    .rename(columns={'index': 'disc_row'})
    # split into list of strings, explode into one string per row
    .assign(token_index=lambda df: df['predictionstring'].str.split(' '))
    .explode('token_index')
    # cast split strings to int, compute position in discourse, assign B-/I- label
    .assign(
        token_index=lambda df: df['token_index'].astype(int),
        pos_in_discourse=lambda df: df.groupby('disc_row').cumcount(),
        label=lambda df: (
            ('B-' + df['discourse_type'])
            .where(df['pos_in_discourse'] == 0,
                   'I-' + df['discourse_type'])
        )
    )
    [['id', 'token_index', 'label']]
)

# 3. Merge tokens with labels, fill missing with 'O'
tokenized = (
    tokenized
    .merge(labels, on=['id', 'token_index'], how='left')
)
tokenized = tokenized.assign(label=lambda df: df['label'].fillna('O'))

# 4. Aggregate back to list of labels per essay
entities_df = (
    tokenized
    .groupby('id')['label']
    .agg(list)
    .reset_index()
    .rename(columns={'label': 'entities'})
)

# 5. Attach entities back to original DataFrame
train_text_df = train_text_df.merge(entities_df, on='id', how='left')

CPU times: user 155 ms, sys: 51.9 ms, total: 207 ms
Wall time: 207 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_26/checkpoints/post_cell_26_try_9.pickle

migration speed (bps): 711077824.9558479
---------------------------
variables to migrate:
plot_ngram 144
dt 57
ax 48
word_dict 589920
other_words_to_take_out 152
train_first 48
len_dict 589920
stop_english 1688
train_first_last 48
i_1 28
text 408
train_text_df 48
train_last 48
plt 72
warnings 72
stopwords 48
pd 72
labels 48
i_3 28
style 72
CountVectorizer 1072
cols_to_merge 120
test_names 126104
f 65
train_texts 126104
IREWR_plug_2 61
counts_dict 696
print_colored_essay 144
glob 144
add_gap_rows_2 144
sample_submission 48
IREWR_tmp 48
test_txt 120
FuncFormatter 1072
sample_discourse_id 32
last_ones 48
ax1 48
av_per_essay 48
keys 120
ax2 48
get_n_grams_2 144
trigrams 48
IREWR_tmp2 48
plot_ngram_2 144
df1 48
np 72
get_n_grams 144
tokenized 48
IREWR_plug_1 61
tqdm 1072
cols_to_display 120
train_txt 126104
myword 28
train 48
total_gaps 48
txt_file 208
all_gaps 48
df 48
mylen 28
add_gap_rows 144
data 2813
t 166
bigrams 48
entities_df 48
myid 61
counter 28
---------------------------
variab

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train_txt', 'stopwords', 'train', 'sample_submission']
Intermediate variables ['test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['sample_discourse_id', 'train']
Intermediate variables ['sample_submission']
Future variables ['train_txt', 'train']
Modified dataframes
  train
    Input columns: set()
    Changed columns: {'discourse_type_num', 'discourse_text', 'id', 'predictionstring', 'discourse_type', 'discourse_end', 'discourse_id', 'discourse_start'}
    Created columns: set()
    Deleted columns: set()
  sample_submission
    Input columns: set()
    Changed columns: {'id', 'class', 'predictionstring'}
    Created columns: set()
    Deleted columns: set()
======= Cell 2 =======
Input variables []
Active variables ['train', 'cols_to_display']
Intermediate variables []
Future variables ['train_txt', 'sample_discourse_id', 'train']
Modified dataframes
  train
    I

In [7]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_26_try_9.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[26], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
counts_dict_opt = counts_dict
train_text_df_opt = train_text_df
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_26.pickle
assert counts_dict_opt == counts_dict
assert compare_df(train_text_df_opt, train_text_df)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
